In [3]:
import pandas as pd
import numpy as np
import os
from scipy import stats


In [4]:
list_files = os.listdir("../")
list_files

['.git',
 'away_team.csv',
 'away_team_score.csv',
 'event.csv',
 'home_team.csv',
 'home_team_score.csv',
 'Javadi',
 'notebook.ipynb',
 'odds.csv',
 'pbp.csv',
 'power.csv',
 'round.csv',
 'season.csv',
 'statistics.csv',
 'time.csv',
 'tournament.csv',
 'venue.csv',
 'votes.csv']

In [5]:
df_statistics = pd.read_csv("../statistics.csv")
home_score_df = pd.read_csv("../home_team_score.csv")
away_score_df = pd.read_csv("../away_team_score.csv")
event_df = pd.read_csv("../event.csv")
home_team_df = pd.read_csv("../home_team.csv")
away_team_df = pd.read_csv("../away_team.csv")


In [6]:

# ─────────────────────────────────────────────────────────────────────────────
# THRESHOLDS — grounded in real tennis records
# Per player per match: 0–35 (buffer above all-time record of 31)
# Combined per match:   0–70
# ─────────────────────────────────────────────────────────────────────────────
MAX_DF_PER_PLAYER = 35
MAX_DF_COMBINED   = 70
MIN_DF            = 0

report         = []
total_original = len(df_statistics)

def log(step, desc, removed, note=''):
    report.append({
        'Step'          : step,
        'Description'   : desc,
        'Rows Removed'  : removed,
        '% of Original' : round(removed / total_original * 100, 2),
        'Note'          : note
    })

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Filter to double_faults + period = 'ALL' only
# ─────────────────────────────────────────────────────────────────────────────
df       = df_statistics.copy()
before   = len(df)
df       = df[
    (df['statistic_name'] == 'double_faults') &
    (df['period'] == 'ALL')
].copy()
removed  = before - len(df)
log(1, "Kept only statistic_name='double_faults' AND period='ALL'", removed,
    'Prevents per-set rows from double counting')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Exact duplicate rows
# ─────────────────────────────────────────────────────────────────────────────
before  = len(df)
df      = df.drop_duplicates()
removed = before - len(df)
log(2, 'Exact duplicate rows', removed, '')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: Duplicate match_ids — all corrupt
# Unlike MatchTimeInfo, statistics ALL-period rows should be exactly one per
# match. Multiple rows = corrupt data, not interrupted matches
# ─────────────────────────────────────────────────────────────────────────────
before    = len(df)
n_corrupt = df[df['match_id'].duplicated(keep=False)]['match_id'].nunique()
df        = df[~df['match_id'].duplicated(keep=False)].copy()
removed   = before - len(df)
log(3, 'Corrupt duplicate match_ids (multiple ALL-period DF rows)',
    removed, f'{n_corrupt} match_ids affected')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: Cross-validate home_value vs home_stat string
# home_stat stores the same value as a string — null out if they disagree
# ─────────────────────────────────────────────────────────────────────────────
home_stat_num = pd.to_numeric(df['home_stat'], errors='coerce')
away_stat_num = pd.to_numeric(df['away_stat'], errors='coerce')

home_mismatch = (
    home_stat_num.notna() & df['home_value'].notna() &
    (home_stat_num != df['home_value'])
)
away_mismatch = (
    away_stat_num.notna() & df['away_value'].notna() &
    (away_stat_num != df['away_value'])
)
cells_nulled = home_mismatch.sum() + away_mismatch.sum()
df.loc[home_mismatch, 'home_value'] = np.nan
df.loc[away_mismatch, 'away_value'] = np.nan

report.append({
    'Step'          : 4,
    'Description'   : 'home/away_value nulled where string stat disagrees',
    'Rows Removed'  : f'{cells_nulled} cells nulled',
    '% of Original' : round(cells_nulled / total_original * 100, 2),
    'Note'          : 'Uses home_stat/away_stat as ground truth'
})

# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: Recover nulls from string columns where possible
# ─────────────────────────────────────────────────────────────────────────────
filled_home = df['home_value'].isna() & home_stat_num.notna()
filled_away = df['away_value'].isna() & away_stat_num.notna()
df.loc[filled_home, 'home_value'] = home_stat_num[filled_home]
df.loc[filled_away, 'away_value'] = away_stat_num[filled_away]

report.append({
    'Step'          : 5,
    'Description'   : 'Null values recovered from home_stat/away_stat strings',
    'Rows Removed'  : f'{filled_home.sum() + filled_away.sum()} cells recovered',
    '% of Original' : '-',
    'Note'          : 'Recovery step'
})

# ─────────────────────────────────────────────────────────────────────────────
# STEP 6: Drop rows where BOTH home and away values are null
# ─────────────────────────────────────────────────────────────────────────────
before  = len(df)
df      = df[df['home_value'].notna() | df['away_value'].notna()].copy()
removed = before - len(df)
log(6, 'Rows where both home_value and away_value are null', removed, '')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 7: Negative double fault counts — impossible
# ─────────────────────────────────────────────────────────────────────────────
before   = len(df)
neg_mask = (
    (df['home_value'].notna() & (df['home_value'] < MIN_DF)) |
    (df['away_value'].notna() & (df['away_value'] < MIN_DF))
)
df       = df[~neg_mask].copy()
removed  = before - len(df)
log(7, 'Rows with negative double fault counts', removed, '')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 8: Unrealistically high per-player double fault counts
# Ceiling = 35 (buffer above all-time record of 31 by Kournikova)
# ─────────────────────────────────────────────────────────────────────────────
before    = len(df)
high_mask = (
    (df['home_value'].notna() & (df['home_value'] > MAX_DF_PER_PLAYER)) |
    (df['away_value'].notna() & (df['away_value'] > MAX_DF_PER_PLAYER))
)
df        = df[~high_mask].copy()
removed   = before - len(df)
log(8, f'Rows where a player exceeded {MAX_DF_PER_PLAYER} double faults',
    removed, 'All-time record is 31 (Kournikova)')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 9: Unrealistically high combined double faults
# ─────────────────────────────────────────────────────────────────────────────
df['total_df'] = df['home_value'].fillna(0) + df['away_value'].fillna(0)
before         = len(df)
df             = df[df['total_df'] <= MAX_DF_COMBINED].copy()
removed        = before - len(df)
log(9, f'Rows where combined double faults exceed {MAX_DF_COMBINED}', removed, '')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 10: Cross-validate against event_df — only finished matches
# ─────────────────────────────────────────────────────────────────────────────
before    = len(df)
valid_ids = event_df[event_df['winner_code'].isin([1, 2])]['match_id']
df        = df[df['match_id'].isin(valid_ids)].copy()
removed   = before - len(df)
log(10, 'Matches not finished or not in event_df', removed, '')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 11: Merge gender from player tables
# Need to know if each match is ATP (M) or WTA (F)
# Use home player gender — already validated in prior questions that
# home and away genders match within a match
# ─────────────────────────────────────────────────────────────────────────────

# Clean player tables
home_clean = (
    home_team_df[['match_id', 'gender']]
    .drop_duplicates(subset='match_id')
    .copy()
)
away_clean = (
    away_team_df[['match_id', 'gender']]
    .drop_duplicates(subset='match_id')
    .copy()
)

# Merge gender from home table first, fall back to away if missing
df = df.merge(
    home_clean.rename(columns={'gender': 'home_gender'}),
    on='match_id', how='left'
)
df = df.merge(
    away_clean.rename(columns={'gender': 'away_gender'}),
    on='match_id', how='left'
)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 12: Drop matches where genders mismatch between home and away
# ─────────────────────────────────────────────────────────────────────────────
before = len(df)
both_known    = df['home_gender'].notna() & df['away_gender'].notna()
gender_clash  = both_known & (df['home_gender'] != df['away_gender'])
df            = df[~gender_clash].copy()
removed       = before - len(df)
log(12, 'Matches with home/away gender mismatch', removed,
    'Cannot mix ATP and WTA rankings')

# Use home_gender as the match gender; fall back to away_gender if missing
df['gender'] = df['home_gender'].fillna(df['away_gender'])

# ─────────────────────────────────────────────────────────────────────────────
# STEP 13: Drop matches with no gender info at all
# ─────────────────────────────────────────────────────────────────────────────
before  = len(df)
df      = df[df['gender'].notna()].copy()
removed = before - len(df)
log(13, 'Matches with no gender information', removed, '')

# Standardise gender labels
df['gender'] = df['gender'].str.strip().str.upper()
df['gender'] = df['gender'].map(
    lambda x: 'M' if x in ['M', 'MALE', 'MEN', '1', 'ATP']
    else ('F' if x in ['F', 'FEMALE', 'WOMEN', '2', 'WTA']
    else np.nan)
)

before  = len(df)
df      = df[df['gender'].notna()].copy()
removed = before - len(df)
log(13, 'Matches with unrecognised gender label after standardisation',
    removed, 'Expected M/F/Male/Female/Men/Women/1/2/ATP/WTA')

# ─────────────────────────────────────────────────────────────────────────────
# FINAL: per-match double faults
# We have home_value and away_value separately so we can analyse:
# 1. Total DFs per match (home + away)
# 2. Per-player DFs (treating each player row independently)
# 3. Split by gender
# ─────────────────────────────────────────────────────────────────────────────
df['total_df'] = df['home_value'] + df['away_value']

# Long format: one row per player per match for per-player analysis
df_long = pd.melt(
    df[['match_id', 'gender', 'home_value', 'away_value']],
    id_vars    = ['match_id', 'gender'],
    value_vars = ['home_value', 'away_value'],
    var_name   = 'side',
    value_name = 'double_faults'
).dropna(subset=['double_faults'])

# ─────────────────────────────────────────────────────────────────────────────
# DESCRIPTIVE STATISTICS
# ─────────────────────────────────────────────────────────────────────────────
def describe_group(data, label):
    desc = data.describe(percentiles=[0.25, 0.5, 0.75])
    print(f"\n{'─' * 60}")
    print(f"  {label}  (n={int(desc['count']):,})")
    print(f"{'─' * 60}")
    print(f"  Mean    : {desc['mean']:.4f}")
    print(f"  Std Dev : {desc['std']:.4f}")
    print(f"  Min     : {desc['min']:.0f}")
    print(f"  25%     : {desc['25%']:.2f}")
    print(f"  Median  : {desc['50%']:.2f}")
    print(f"  75%     : {desc['75%']:.2f}")
    print(f"  Max     : {desc['max']:.0f}")
    print(f"  Skew    : {data.skew():.4f}")
    print(f"  Kurt    : {data.kurt():.4f}")
    return desc

# ─────────────────────────────────────────────────────────────────────────────
# STATISTICAL SIGNIFICANCE TEST
# Mann-Whitney U (non-parametric) — double faults are count data,
# likely right-skewed, so we don't assume normality
# ─────────────────────────────────────────────────────────────────────────────
atp_df  = df_long[df_long['gender'] == 'M']['double_faults']
wta_df  = df_long[df_long['gender'] == 'F']['double_faults']

# Total per match by gender (for match-level analysis)
atp_total = df[df['gender'] == 'M']['total_df']
wta_total = df[df['gender'] == 'F']['total_df']

u_stat_player, p_val_player = stats.mannwhitneyu(
    atp_df, wta_df, alternative='two-sided'
)
u_stat_match, p_val_match = stats.mannwhitneyu(
    atp_total, wta_total, alternative='two-sided'
)

# Effect size — rank-biserial correlation
def rank_biserial(u, n1, n2):
    return 1 - (2 * u) / (n1 * n2)

rb_player = rank_biserial(u_stat_player, len(atp_df), len(wta_df))
rb_match  = rank_biserial(u_stat_match,  len(atp_total), len(wta_total))

# ─────────────────────────────────────────────────────────────────────────────
# PRINT FULL REPORT
# ─────────────────────────────────────────────────────────────────────────────
total_clean   = len(df)
total_removed = total_original - total_clean

print("=" * 60)
print("DATA CLEANING REPORT — Double Faults by Gender")
print("=" * 60)
print(f"  Original rows       : {total_original:,}")
print(f"  Rows after cleaning : {total_clean:,}")
print(f"  Total removed       : {total_removed:,} ({round(total_removed/total_original*100,2)}%)")
print(f"  Clean data          : {round(total_clean/total_original*100,2)}%")
print("=" * 60)
print(pd.DataFrame(report).to_string(index=False))







DATA CLEANING REPORT — Double Faults by Gender
  Original rows       : 1,358,234
  Rows after cleaning : 9,620
  Total removed       : 1,348,614 (99.29%)
  Clean data          : 0.71%
 Step                                                  Description        Rows Removed % of Original                                           Note
    1    Kept only statistic_name='double_faults' AND period='ALL'             1334951         98.29     Prevents per-set rows from double counting
    2                                         Exact duplicate rows               10442          0.77                                               
    3    Corrupt duplicate match_ids (multiple ALL-period DF rows)                2834          0.21                        1382 match_ids affected
    4           home/away_value nulled where string stat disagrees    160 cells nulled          0.01       Uses home_stat/away_stat as ground truth
    5       Null values recovered from home_stat/away_stat strings 160 cells

In [7]:

print("\n\n" + "=" * 60)
print("DESCRIPTIVE STATISTICS — Double Faults per Player per Match")
print("=" * 60)
describe_group(atp_df,  'ATP (Men)')
describe_group(wta_df,  'WTA (Women)')




DESCRIPTIVE STATISTICS — Double Faults per Player per Match

────────────────────────────────────────────────────────────
  ATP (Men)  (n=10,890)
────────────────────────────────────────────────────────────
  Mean    : 2.7328
  Std Dev : 2.2409
  Min     : 0
  25%     : 1.00
  Median  : 2.00
  75%     : 4.00
  Max     : 21
  Skew    : 1.3272
  Kurt    : 3.0883

────────────────────────────────────────────────────────────
  WTA (Women)  (n=8,350)
────────────────────────────────────────────────────────────
  Mean    : 3.5726
  Std Dev : 2.7719
  Min     : 0
  25%     : 2.00
  Median  : 3.00
  75%     : 5.00
  Max     : 28
  Skew    : 1.3451
  Kurt    : 3.1049


count    8350.000000
mean        3.572575
std         2.771912
min         0.000000
25%         2.000000
50%         3.000000
75%         5.000000
max        28.000000
Name: double_faults, dtype: float64

In [8]:
print("\n\n" + "=" * 60)
print("DESCRIPTIVE STATISTICS — Total Double Faults per Match")
print("=" * 60)
describe_group(atp_total, 'ATP Matches — Combined DFs')
describe_group(wta_total, 'WTA Matches — Combined DFs')

print("\n\n" + "=" * 60)
print("STATISTICAL SIGNIFICANCE — Is the difference real?")
print("=" * 60)
print(f"\n  Test: Mann-Whitney U (non-parametric, no normality assumed)")
print(f"  Reason: double fault counts are right-skewed count data\n")



DESCRIPTIVE STATISTICS — Total Double Faults per Match

────────────────────────────────────────────────────────────
  ATP Matches — Combined DFs  (n=5,445)
────────────────────────────────────────────────────────────
  Mean    : 5.4656
  Std Dev : 3.4076
  Min     : 0
  25%     : 3.00
  Median  : 5.00
  75%     : 7.00
  Max     : 25
  Skew    : 1.0875
  Kurt    : 1.9621

────────────────────────────────────────────────────────────
  WTA Matches — Combined DFs  (n=4,175)
────────────────────────────────────────────────────────────
  Mean    : 7.1451
  Std Dev : 4.2634
  Min     : 0
  25%     : 4.00
  Median  : 6.00
  75%     : 9.00
  Max     : 36
  Skew    : 1.0939
  Kurt    : 1.9700


STATISTICAL SIGNIFICANCE — Is the difference real?

  Test: Mann-Whitney U (non-parametric, no normality assumed)
  Reason: double fault counts are right-skewed count data



In [9]:
print(f"  Per-player analysis:")
print(f"    U statistic : {u_stat_player:.2f}")
print(f"    p-value     : {p_val_player:.6f}")
print(f"    Effect size : {rb_player:.4f}  (rank-biserial correlation)")
print(f"    Significant : {'YES ✓' if p_val_player < 0.05 else 'NO ✗'}  (α = 0.05)")

print(f"\n  Per-match (combined) analysis:")
print(f"    U statistic : {u_stat_match:.2f}")
print(f"    p-value     : {p_val_match:.6f}")
print(f"    Effect size : {rb_match:.4f}  (rank-biserial correlation)")
print(f"    Significant : {'YES ✓' if p_val_match < 0.05 else 'NO ✗'}  (α = 0.05)")

  Per-player analysis:
    U statistic : 37319504.50
    p-value     : 0.000000
    Effect size : 0.1792  (rank-biserial correlation)
    Significant : YES ✓  (α = 0.05)

  Per-match (combined) analysis:
    U statistic : 8648315.50
    p-value     : 0.000000
    Effect size : 0.2391  (rank-biserial correlation)
    Significant : YES ✓  (α = 0.05)


In [10]:
print(f"\n  Effect size interpretation (rank-biserial):")
print(f"    |r| < 0.1  → negligible")
print(f"    |r| < 0.3  → small")
print(f"    |r| < 0.5  → medium")
print(f"    |r| >= 0.5 → large")

print(f"\n  ATP mean DFs/player : {atp_df.mean():.4f}")
print(f"  WTA mean DFs/player : {wta_df.mean():.4f}")
print(f"  Difference          : {abs(atp_df.mean() - wta_df.mean()):.4f}")
print(f"  WTA vs ATP ratio    : {wta_df.mean() / atp_df.mean():.4f}x")


  Effect size interpretation (rank-biserial):
    |r| < 0.1  → negligible
    |r| < 0.3  → small
    |r| < 0.5  → medium
    |r| >= 0.5 → large

  ATP mean DFs/player : 2.7328
  WTA mean DFs/player : 3.5726
  Difference          : 0.8398
  WTA vs ATP ratio    : 1.3073x
